# 🔍 The Outer Gap Diagnostic
### EPS Research High-School Exploration Track — Ages 15-18

The **outer gap** is $V_{adj}(R_2)-V_{bary}(R_2)$: the residual between
the omega-corrected velocity and the baryonic velocity at the outermost ring.

This notebook reproduces the **frozen 84-galaxy SPARC cohort** used in the
published analysis. The corrected FAIR² calculation converts stored omega
values from rad/Gyr back to native km/s/kpc before velocity arithmetic.

The corrected reproduction finds all 84 outer gaps negative, with a mean
of about **−55.3 ± 28.6 km/s**. The archived Paper 2 analysis reported
−51.4 ± 25.0 km/s; the numerical value changes after the unit correction,
while the qualitative negative-sign result remains.

This is a **boundary consistency diagnostic**. It should not be interpreted
as an independent exclusion of an NFW or dark-matter halo model.

**Prerequisites:** Understanding of residuals, basic RMSE

In [ ]:
# ── Colab setup: canonical FAIR² corpus paths ─────────────
import os, sys
IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    import urllib.request
    CORPORA = {
        'rotation_curve_corpus_v7.json': 'https://zenodo.org/records/19563417/files/rotation_curve_corpus_v7.json',
        'high_z_kinematic_corpus_Z1.json': 'https://zenodo.org/records/21834678/files/high_z_kinematic_corpus_Z1.json',
        'dwarf_irregular_corpus_v1.json': 'https://zenodo.org/records/20320362/files/dwarf_irregular_corpus_v1.json',
    }
    for filename, url in CORPORA.items():
        if not os.path.exists(filename):
            print(f"Downloading {filename}...")
            urllib.request.urlretrieve(url, filename)
            print(f"  ✓ {filename}")
        else:
            print(f"  Already present: {filename}")

    HI_PATH = 'rotation_curve_corpus_v7.json'
    Z1_PATH = 'high_z_kinematic_corpus_Z1.json'
    DWARF_PATH = 'dwarf_irregular_corpus_v1.json'
    print("Ready.")
else:
    HI_PATH = '../hi/rotation_curve_corpus_v7.json'
    Z1_PATH = '../highz/high_z_kinematic_corpus_Z1.json'
    DWARF_PATH = '../dwarfs/dwarf_irregular_corpus_v1.json'
    print("Running locally — using canonical repository corpus paths.")


In [ ]:
import matplotlib
matplotlib.use('Agg')
import json, numpy as np, matplotlib.pyplot as plt

# Frozen published SPARC cohort and Paper2 parameters.
# omega is stored/reported in rad/Gyr; velocity arithmetic uses km/s/kpc.
P2_DATA = [{'galaxy': 'NGC3741', 'omega': 7.032, 'upsilon': 1.0},
 {'galaxy': 'UGC08550', 'omega': 9.317, 'upsilon': 1.0},
 {'galaxy': 'NGC3109', 'omega': 10.244, 'upsilon': 1.0},
 {'galaxy': 'UGC07603', 'omega': 14.2, 'upsilon': 0.94},
 {'galaxy': 'DDO064', 'omega': 15.35, 'upsilon': 1.0},
 {'galaxy': 'UGC01281', 'omega': 11.37, 'upsilon': 1.0},
 {'galaxy': 'UGC07151', 'omega': 12.53, 'upsilon': 0.953},
 {'galaxy': 'UGC07399', 'omega': 14.86, 'upsilon': 1.0},
 {'galaxy': 'UGC04278', 'omega': 13.76, 'upsilon': 1.0},
 {'galaxy': 'NGC3972', 'omega': 13.93, 'upsilon': 0.675},
 {'galaxy': 'NGC7793', 'omega': 11.4, 'upsilon': 0.111},
 {'galaxy': 'F571-8', 'omega': 9.17, 'upsilon': 0.104},
 {'galaxy': 'UGC05721', 'omega': 11.51, 'upsilon': 0.979},
 {'galaxy': 'UGC07323', 'omega': 13.41, 'upsilon': 0.986},
 {'galaxy': 'NGC3521', 'omega': 10.25, 'upsilon': 0.535},
 {'galaxy': 'F563-V2', 'omega': 11.08, 'upsilon': 1.0},
 {'galaxy': 'ESO116-G012', 'omega': 11.02, 'upsilon': 1.0},
 {'galaxy': 'F568-1', 'omega': 10.52, 'upsilon': 1.0},
 {'galaxy': 'ESO079-G014', 'omega': 10.37, 'upsilon': 0.742},
 {'galaxy': 'NGC3893', 'omega': 6.47, 'upsilon': 0.507},
 {'galaxy': 'UGC08286', 'omega': 9.82, 'upsilon': 1.0},
 {'galaxy': 'NGC0024', 'omega': 9.55, 'upsilon': 0.873},
 {'galaxy': 'NGC0100', 'omega': 9.37, 'upsilon': 0.504},
 {'galaxy': 'NGC0891', 'omega': 8.99, 'upsilon': 0.326},
 {'galaxy': 'NGC4217', 'omega': 9.99, 'upsilon': 0.1},
 {'galaxy': 'UGC06917', 'omega': 8.3, 'upsilon': 1.0},
 {'galaxy': 'NGC7814', 'omega': 8.52, 'upsilon': 0.582},
 {'galaxy': 'NGC3917', 'omega': 8.82, 'upsilon': 0.465},
 {'galaxy': 'F583-4', 'omega': 9.33, 'upsilon': 1.0},
 {'galaxy': 'IC4202', 'omega': 9.3, 'upsilon': 0.1},
 {'galaxy': 'UGC08490', 'omega': 6.95, 'upsilon': 1.0},
 {'galaxy': 'NGC4088', 'omega': 6.98, 'upsilon': 0.383},
 {'galaxy': 'NGC6946', 'omega': 6.83, 'upsilon': 0.426},
 {'galaxy': 'F568-3', 'omega': 6.54, 'upsilon': 0.829},
 {'galaxy': 'F568-V1', 'omega': 6.55, 'upsilon': 1.0},
 {'galaxy': 'NGC2403', 'omega': 6.32, 'upsilon': 0.84},
 {'galaxy': 'UGC12632', 'omega': 6.27, 'upsilon': 1.0},
 {'galaxy': 'UGC11455', 'omega': 6.24, 'upsilon': 0.243},
 {'galaxy': 'NGC5985', 'omega': 6.16, 'upsilon': 1.0},
 {'galaxy': 'UGC00731', 'omega': 5.99, 'upsilon': 1.0},
 {'galaxy': 'UGC06786', 'omega': 5.87, 'upsilon': 0.736},
 {'galaxy': 'NGC6195', 'omega': 5.83, 'upsilon': 0.52},
 {'galaxy': 'UGC12732', 'omega': 5.81, 'upsilon': 1.0},
 {'galaxy': 'NGC4157', 'omega': 5.46, 'upsilon': 0.423},
 {'galaxy': 'UGC06930', 'omega': 5.44, 'upsilon': 1.0},
 {'galaxy': 'UGC03205', 'omega': 5.27, 'upsilon': 0.65},
 {'galaxy': 'UGC11820', 'omega': 5.21, 'upsilon': 1.0},
 {'galaxy': 'NGC4100', 'omega': 5.2, 'upsilon': 0.867},
 {'galaxy': 'UGC03546', 'omega': 5.29, 'upsilon': 0.433},
 {'galaxy': 'F563-1', 'omega': 5.02, 'upsilon': 1.0},
 {'galaxy': 'NGC4559', 'omega': 5.3, 'upsilon': 0.551},
 {'galaxy': 'F583-1', 'omega': 5.16, 'upsilon': 0.677},
 {'galaxy': 'NGC2955', 'omega': 5.71, 'upsilon': 0.564},
 {'galaxy': 'NGC7331', 'omega': 4.9, 'upsilon': 0.409},
 {'galaxy': 'NGC4183', 'omega': 4.92, 'upsilon': 1.0},
 {'galaxy': 'DDO161', 'omega': 4.69, 'upsilon': 1.0},
 {'galaxy': 'NGC6503', 'omega': 4.3, 'upsilon': 0.59},
 {'galaxy': 'NGC2998', 'omega': 4.61, 'upsilon': 0.859},
 {'galaxy': 'NGC1090', 'omega': 5.24, 'upsilon': 0.184},
 {'galaxy': 'NGC5033', 'omega': 3.79, 'upsilon': 0.49},
 {'galaxy': 'NGC5371', 'omega': 3.78, 'upsilon': 0.617},
 {'galaxy': 'F579-V1', 'omega': 7.13, 'upsilon': 1.0},
 {'galaxy': 'UGC06983', 'omega': 5.74, 'upsilon': 0.978},
 {'galaxy': 'NGC2841', 'omega': 3.58, 'upsilon': 0.966},
 {'galaxy': 'UGC05750', 'omega': 3.44, 'upsilon': 0.525},
 {'galaxy': 'UGC05005', 'omega': 3.38, 'upsilon': 0.806},
 {'galaxy': 'UGC02885', 'omega': 3.4, 'upsilon': 0.823},
 {'galaxy': 'NGC5055', 'omega': 2.89, 'upsilon': 0.385},
 {'galaxy': 'NGC6674', 'omega': 2.81, 'upsilon': 0.953},
 {'galaxy': 'UGC01230', 'omega': 2.74, 'upsilon': 0.863},
 {'galaxy': 'UGC06614', 'omega': 2.49, 'upsilon': 0.597},
 {'galaxy': 'UGC02487', 'omega': 2.49, 'upsilon': 1.0},
 {'galaxy': 'UGC00128', 'omega': 2.23, 'upsilon': 1.0},
 {'galaxy': 'NGC0801', 'omega': 3.32, 'upsilon': 0.448},
 {'galaxy': 'UGC09133', 'omega': 1.97, 'upsilon': 0.66},
 {'galaxy': 'UGC07125', 'omega': 3.09, 'upsilon': 0.787},
 {'galaxy': 'NGC1003', 'omega': 3.49, 'upsilon': 0.679},
 {'galaxy': 'NGC3198', 'omega': 3.33, 'upsilon': 0.149},
 {'galaxy': 'NGC2903', 'omega': 7.01, 'upsilon': 0.1},
 {'galaxy': 'ESO563-G021', 'omega': 7.31, 'upsilon': 0.1},
 {'galaxy': 'NGC5585', 'omega': 8.06, 'upsilon': 0.6},
 {'galaxy': 'F574-1', 'omega': 7.68, 'upsilon': 1.0},
 {'galaxy': 'UGC06446', 'omega': 7.11, 'upsilon': 1.0},
 {'galaxy': 'UGC07524', 'omega': 7.15, 'upsilon': 0.87}]

assert len(P2_DATA) == 84
assert len({d['galaxy'] for d in P2_DATA}) == 84

with open(HI_PATH) as f:
    corpus = json.load(f)

# Restrict identity lookup to SPARC so duplicate cross-survey names cannot overwrite it.
galaxies = {
    g['galaxy']: g
    for g in corpus['galaxies']
    if g.get('survey') == 'SPARC'
}

UPSILON_BUL = 0.7
gaps = []
missing = []

for d in P2_DATA:
    name = d['galaxy']
    if name not in galaxies:
        missing.append(name)
        continue

    g = galaxies[name]
    omega_rad_gyr = d['omega']
    omega_kms_kpc = omega_rad_gyr / 1.0227
    upsilon = d['upsilon']

    data = [
        p for p in g.get('data', [])
        if p.get('Vobs', 0) > 0 and p.get('Rad', 0) > 0
    ]
    if len(data) < 2:
        missing.append(name)
        continue

    p2 = data[-1]
    R2    = p2['Rad']
    Vobs2 = p2['Vobs']
    Vgas2 = p2.get('Vgas', 0)
    Vdsk2 = p2.get('Vdisk', 0)
    Vbul2 = p2.get('Vbul', 0)

    # Paper2 baryonic convention: sign-preserving gas term,
    # per-galaxy disk upsilon, fixed bulge upsilon=0.7.
    Vbar2_sq = (
        abs(Vgas2) * Vgas2
        + upsilon * Vdsk2**2
        + UPSILON_BUL * Vbul2**2
    )
    if Vbar2_sq <= 0:
        missing.append(name)
        continue

    Vbary2 = Vbar2_sq**0.5

    # LOCKED UNIT INVARIANT:
    # velocity arithmetic uses omega in native km/s/kpc.
    Vadj2 = Vobs2 - R2 * omega_kms_kpc
    gap = Vadj2 - Vbary2

    gaps.append({
        'galaxy': name,
        'gap': gap,
        'Vadj': Vadj2,
        'Vbary': Vbary2,
        'vmax': max(p['Vobs'] for p in data)
    })

assert not missing, f"Frozen cohort missing/invalid: {missing}"
assert len(gaps) == 84

gaps_arr = np.array([g['gap'] for g in gaps])
all_neg = int(np.sum(gaps_arr < 0))
mean_gap = np.mean(gaps_arr)
std_gap = np.std(gaps_arr)
median_gap = np.median(gaps_arr)

print(f'Frozen SPARC cohort: {len(gaps)}/84')
print(f'Negative outer gaps: {all_neg}/{len(gaps)}')
print(f'Mean outer gap: {mean_gap:.1f} ± {std_gap:.1f} km/s')
print(f'Median outer gap: {median_gap:.1f} km/s')
print('Archived Paper 2: −51.4 ± 25.0 km/s, all 84 negative')
print('Corrected FAIR² reproduction uses native km/s/kpc for velocity arithmetic.')

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

ax = axes[0]
ax.hist(gaps_arr, bins=20, alpha=0.8, edgecolor='white')
ax.axvline(0, lw=2, ls='--', label='Zero')
ax.axvline(mean_gap, lw=2, label=f'Mean = {mean_gap:.1f} km/s')
ax.set_xlabel('Outer Gap: Vadj(R2) − Vbary(R2) (km/s)', fontsize=11)
ax.set_ylabel('N galaxies', fontsize=11)
ax.set_title('Frozen N=84 SPARC Cohort', fontsize=11)
ax.legend(fontsize=9)

ax2 = axes[1]
ax2.scatter([g['vmax'] for g in gaps], gaps_arr, s=18, alpha=0.6)
ax2.axhline(0, lw=1.5, ls='--')
ax2.set_xlabel('Vmax (km/s)', fontsize=11)
ax2.set_ylabel('Outer Gap (km/s)', fontsize=11)
ax2.set_title('Outer Gap vs Rotation Speed', fontsize=11)
ax2.grid(alpha=0.3)

plt.suptitle(
    '🔍 Outer Gap Diagnostic — Corrected FAIR² Reproduction\n'
    'Boundary consistency diagnostic; not an independent halo exclusion',
    fontsize=11
)
plt.tight_layout()
plt.savefig('hs_b_05_outer_gap.png', dpi=150, bbox_inches='tight')
plt.show()
